# Thực nghiệm cuối kỳ T2f
Mở notebook từ dự án đã chuẩn bị dữ liệu, hoặc thêm gói đầy đủ `t2f-final.zip` vào Kaggle/Colab.

In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

kaggle = Path("/kaggle/input")
work = Path("/kaggle/working") if kaggle.exists() else Path.cwd()
roots = [Path.cwd(), Path.cwd().parent, work / "t2f-forecasting"]
repo = next((root for root in roots if (root / "final/compare.py").is_file()), None)

if repo is None:
    script = next(kaggle.rglob("final/compare.py"), None)
    if script is not None:
        repo = script.parent.parent

if repo is None:
    archive = next((p for root in [work, kaggle] for p in root.rglob("t2f-final.zip")), None)
    if archive is None and "google.colab" in sys.modules:
        from google.colab import files

        archive = Path(next(iter(files.upload())))
    if archive is None:
        raise FileNotFoundError("Cần thêm gói t2f-final.zip có dữ liệu.")
    with zipfile.ZipFile(archive) as bundle:
        bundle.extractall(work)
    repo = work / "t2f-forecasting"

print(repo)

In [ ]:
datasets = ["ETTh", "ETTm", "ELD"]
bridge = True
long_horizon = True
run_td3 = True
run_patch = True
audit = False
output = work / "final_runs"
env = {**os.environ, "OPENBLAS_NUM_THREADS": "1", "OMP_NUM_THREADS": "1"}


def run(module, *args, destination=output):
    command = [sys.executable, "-m", f"final.{module}", *args, "--output", str(destination)]
    subprocess.run(command, cwd=repo, env=env, check=True)


for dataset in datasets:
    args = ["--dataset", dataset]
    run("compare", *args)
    if bridge:
        run("compare", *args, "--bridge")
    if run_td3:
        run("td3", *args)
    if dataset != "ELD":
        if run_patch:
            run("patch_transformer", *args)
        if long_horizon:
            run("compare", *args, "--horizon", "96")
            if run_patch:
                run("patch_transformer", *args, "--horizon", "96")
    if audit:
        name = "eld" if dataset == "ELD" else dataset
        run("local_audit", "--dataset", name, destination=output / "local_audit")

run("summarize")
print(shutil.make_archive(str(work / "final_results"), "zip", output))